In [1]:
# How do the relative price dynamics between butter and margarine evolve during periods of elevated food inflation?
# Wie verhalten sich Butter und Margarine im Verhältnis zueinander?
# 	Wird Butter relativ teurer als Margarine?
#   Passiert das besonders in Phasen hoher Inflatio

In [2]:
# Analyse in 5 Schritten 

# 1. Butter und Margarine Daten laden und in ein Tabelle umwandeln und zusammenführen 
# 2. Daten bereinigen und auf denselben Zeitraum bringen
# 3. Variable einbauen den Preisabstand (BUTTER - MARGARINE) berechnen
# 4. UNtersuchen der Inflationsphase  
# 5. Grafik erstellen


In [3]:
import json
import pandas as pd


def eurostat_json_to_df(data):
    """
    Convert a Eurostat JSON dataset into a clean monthly DataFrame.

    
    ----------
    data : dict
        Eurostat JSON data.

    Returns
    -------
    pandas.DataFrame
        DataFrame with columns month and value.
    """
    time_index = data["dimension"]["time"]["category"]["index"]
    time_df = pd.DataFrame(list(time_index.items()), columns=["month", "pos"])

    values = data["value"]
    value_df = pd.DataFrame(list(values.items()), columns=["pos", "value"])
    value_df["pos"] = value_df["pos"].astype(int) # position in integer umwandeln

    df = pd.merge(time_df, value_df, on="pos")
    df = df.drop(columns=["pos"])

    return df


with open("eurostat_butter_cpi.json", "r", encoding="utf-8") as file:
    data_butter = json.load(file)

with open("margarine_cpi.json", "r", encoding="utf-8") as file:
    data_margarine = json.load(file)


butter_df = eurostat_json_to_df(data_butter)
margarine_df = eurostat_json_to_df(data_margarine)


print("Butter CPI")
print(butter_df.head())

print("\nMargarine CPI")
print(margarine_df.head())

Butter CPI
     month  value
0  2014-12   99.3
1  2015-01   99.3
2  2015-02   99.0
3  2015-03  105.7
4  2015-04  106.2

Margarine CPI
     month  value
0  2014-12  101.7
1  2015-01  102.7
2  2015-02  101.9
3  2015-03  102.0
4  2015-04  103.4


In [4]:
# Convert month column to datetime
# Monat in echte Datum umwandeln
butter_df["month"] = pd.to_datetime(butter_df["month"])
margarine_df["month"] = pd.to_datetime(margarine_df["month"])

# Rename value columns
butter_df = butter_df.rename(columns={"value": "butter_cpi"})
margarine_df = margarine_df.rename(columns={"value": "margarine_cpi"})

# Sort by month
butter_df = butter_df.sort_values("month").reset_index(drop=True)
margarine_df = margarine_df.sort_values("month").reset_index(drop=True)

#  cleaned tables
print("Cleaned Butter CPI")
print(butter_df.head())

print("\nCleaned Margarine CPI")
print(margarine_df.head())

Cleaned Butter CPI
       month  butter_cpi
0 2014-12-01        99.3
1 2015-01-01        99.3
2 2015-02-01        99.0
3 2015-03-01       105.7
4 2015-04-01       106.2

Cleaned Margarine CPI
       month  margarine_cpi
0 2014-12-01          101.7
1 2015-01-01          102.7
2 2015-02-01          101.9
3 2015-03-01          102.0
4 2015-04-01          103.4


In [5]:
# Anzahl der Zeilen vorher
butter_before = len(butter_df)
margarine_before = len(margarine_df)

# Zeitraum filtern
# 2020 als Vprphase 
# 2021-2022 als Inflationsphase
# 2023 als Umkehr/Nachphase
butter_df = butter_df[(butter_df["month"] >= "2020-01-01") & (butter_df["month"] <= "2023-12-01")]
margarine_df = margarine_df[(margarine_df["month"] >= "2020-01-01") & (margarine_df["month"] <= "2023-12-01")]

# Anzahl der Zeilen nach Zeiraum filtern
butter_afterTime = len(butter_df)
margarine_afterTime = len(margarine_df)

# Fehlende Werte entfernen
butter_df = butter_df.dropna()
margarine_df = margarine_df.dropna()

# Duplikate entfernen
butter_df = butter_df.drop_duplicates()
margarine_df = margarine_df.drop_duplicates()

# Sortieren und Index zurücksetzen
butter_df = butter_df.sort_values("month").reset_index(drop=True)
margarine_df = margarine_df.sort_values("month").reset_index(drop=True)

# Anzahl der Zeilen nachher
butter_after = len(butter_df)
margarine_after = len(margarine_df)

# Ausgabe
print(butter_before, butter_afterTime, butter_after)
print(margarine_before, margarine_afterTime, margarine_after)

133 48 48
133 48 48


In [6]:
# Mergent der Tabellen
# für den direkten Vergleich 
merged_df = butter_df.merge(margarine_df, on="month")

print(merged_df.head())

       month  butter_cpi  margarine_cpi
0 2020-01-01       144.9          110.9
1 2020-02-01       143.3          110.8
2 2020-03-01       142.7          110.6
3 2020-04-01       142.5          111.2
4 2020-05-01       142.3          109.6


In [7]:
#Wir brauchen eine Variable die zeigt, wie groß der Preisunterschied zwischen Butter und Margarine ist
# butter_cpi - margarine_cpi 
# Interpretation: groß: Butter viel teurer als Margarine, klein: Preise näher beieinander, negativ: Margarine teurer 

# Preisabstand berechnen
merged_df["price_gap"] = merged_df["butter_cpi"] - merged_df["margarine_cpi"]

print(merged_df.head())


       month  butter_cpi  margarine_cpi  price_gap
0 2020-01-01       144.9          110.9       34.0
1 2020-02-01       143.3          110.8       32.5
2 2020-03-01       142.7          110.6       32.1
3 2020-04-01       142.5          111.2       31.3
4 2020-05-01       142.3          109.6       32.7


In [8]:
# Inflationsphase markieren 
# 2021-01 bis 2022-12

# boolische Spalte erstellen
merged_df["inflation_period"] = (
    (merged_df["month"] >= "2021-01-01") &
    (merged_df["month"] <= "2022-12-01")
)

print("Start der Daten:")
print(merged_df.head(10))

print("\nBeginn der Inflationsphase:")
print(merged_df[merged_df["month"] >= "2021-01-01"].head(5))

print("\nEnde der Inflationsphase:")
print(merged_df[merged_df["month"] >= "2022-10-01"].head(5))

print("\nEnde der Daten:")
print(merged_df.tail(10))

Start der Daten:
       month  butter_cpi  margarine_cpi  price_gap  inflation_period
0 2020-01-01       144.9          110.9       34.0             False
1 2020-02-01       143.3          110.8       32.5             False
2 2020-03-01       142.7          110.6       32.1             False
3 2020-04-01       142.5          111.2       31.3             False
4 2020-05-01       142.3          109.6       32.7             False
5 2020-06-01       140.3          110.5       29.8             False
6 2020-07-01       140.5          109.2       31.3             False
7 2020-08-01       141.2          109.0       32.2             False
8 2020-09-01       138.2          109.8       28.4             False
9 2020-10-01       136.9          108.0       28.9             False

Beginn der Inflationsphase:
        month  butter_cpi  margarine_cpi  price_gap  inflation_period
12 2021-01-01       142.1          110.4       31.7              True
13 2021-02-01       139.7          109.7       30.0    

In [9]:
# durchschnittlicher Preiabstand vergleichen
inflation_gap = merged_df[merged_df["inflation_period"]]["price_gap"].mean() # Mittelwert des Preisabstands während der Inflationsphase

normal_gap = merged_df[~merged_df["inflation_period"]]["price_gap"].mean() # Mittelwert des Preisabstands außerhalb der Inflationsphase

print("Average price gap during inflation:", inflation_gap) 
print("Average price gap outside inflation:", normal_gap)

# inflation_gap deutlich größer ist als normal_gap
# Butter wurde relativ deutlich teurer als Margarine während der Inflation
# Der durchschnittliche Preisabstand war in der Inflationsphase viel größer.




Average price gap during inflation: 45.104166666666664
Average price gap outside inflation: 10.62916666666667


In [10]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=merged_df["month"],
        y=merged_df["butter_cpi"],
        mode="lines",
        name="Butter CPI",
        fill="tozeroy"
    )
)

fig.add_trace(
    go.Scatter(
        x=merged_df["month"],
        y=merged_df["margarine_cpi"],
        mode="lines",
        name="Margarine CPI",
        fill="tozeroy"
    )
)

fig.update_layout(
    title="Butter and Margarine Price Dynamics during Food Inflation",
    xaxis_title="Month",
    yaxis_title="Price Index (2015 = 100)"
)

fig.show()

# Diese Grafik zeigt die Preisentwicklung von Butter und Margarine über die Zeit.
# Man erkennt, dass beide Produkte während der Inflationsphase steigen,
# Butter jedoch deutlich stärker. 
# Dadurch wird sichtbar, dass sich die Preisentwicklung beider
# Produkte während hoher Inflation auseinanderbewegt.


In [11]:
import plotly.express as px

fig = px.area(
    merged_df,
    x="month",
    y="price_gap",
    title="Price Gap between Butter and Margarine",
    labels={
        "month": "Month",
        "price_gap": "Price Gap (Butter - Margarine)"
    }
)


fig.add_hline(y=0)

fig.show()

In [12]:
'''
Diese Grafik zeigt den Preisabstand zwischen Butter und Margarine.
Positive Werte bedeuten, dass Butter teurer ist als Margarine, 
negative Werte bedeuten, dass Margarine teurer ist. 
Während der Inflationsphase wächst der Abstand stark an, 
danach fällt er wieder und wird 2023 teilweise negativ. 
Dadurch wird die relative Preisdynamik zwischen beiden Produkten direkt sichtbar.
'''


'\nDiese Grafik zeigt den Preisabstand zwischen Butter und Margarine.\nPositive Werte bedeuten, dass Butter teurer ist als Margarine, \nnegative Werte bedeuten, dass Margarine teurer ist. \nWährend der Inflationsphase wächst der Abstand stark an, \ndanach fällt er wieder und wird 2023 teilweise negativ. \nDadurch wird die relative Preisdynamik zwischen beiden Produkten direkt sichtbar.\n'

In [13]:
# Website 
merged_df.to_csv("rq2_data.csv", index=False)